# 00. Setup: 키, 서버, 프로토콜 버전이 준비되었는가

이 노트북은 수업 시작 전 10분 점검입니다. 01부터는 모든 셀이 이 점검이 통과했다고 가정합니다. 여기서 막히면 01로 넘어가지 말고 먼저 풀고 오세요.

## 수업 시간표

| 구간 | 내용 | 분 |
|---|---|---|
| 1 | 커널 위치와 `cap_lab` import 확인 | 2 |
| 2 | 키 설정 상태 확인 | 2 |
| 3 | MCP 서버를 띄우고 Tool 7개 나열 | 2 |
| 4 | MCP 프로토콜 버전 측정 | 1 |
| 5 | Claude 연결 확인 (유료 1회) | 1 |
| 6 | Jev 연결 확인 (유료 1회) | 1 |
| 7 | Langfuse 인증 확인, 직접 해보기, 정리 | 1 |

## 이 노트북이 확인하는 것

Day04_new 과정은 하나의 사례를 끝까지 끌고 갑니다. 고객이 하고 싶은 일을 인터뷰해서 요구사항을 모으는 **MCP 서버**(Tool 7개)를 만들고, `create_deep_agent`와 `ChatAnthropic`의 `claude-sonnet-5-5`(effort `medium`)로 만든 **Deep Agent**가 그 서버를 부립니다. 측정된 병목에서 하는 판단(Decision)은 **Jev**(TypeSafe Decision API)에 맡기고, 측정 기록은 **Langfuse**에 남깁니다.

따라서 수업을 시작하려면 다섯 가지가 준비되어 있어야 합니다.

1. 커널의 작업 폴더가 `Day04_new`이고 `cap_lab` 패키지를 import할 수 있다.
2. 키가 `Day04_new/.env`에 채워져 있다. 값이 아니라 설정됨/미설정만 확인합니다.
3. MCP 서버가 stdio로 뜨고 Tool 7개가 보인다.
4. 클라이언트와 서버가 실제로 쓰는 MCP 프로토콜 버전을 읽을 수 있다.
5. Claude와 Jev를 한 번씩 불러 본다. Langfuse는 인증만 확인한다.

## 호출 규모

이 노트북의 유료 호출은 두 번뿐입니다. Claude 1회(5절), Jev 1회(6절). 유료 셀은 실행 직전에 `announce(...)`로 규모를 한 줄 알립니다. 직접 해보기의 유료 셀은 `RUN_EXERCISE = False`로 잠겨 있고, 켜는 것은 본인의 결정입니다. 이 과정의 모든 코드는 제공되며 점수는 매기지 않습니다. 목표는 코드를 완성하는 것이 아니라 각 셀이 무엇을 하는지 자기 말로 설명하는 것입니다.

## 1. 커널은 어디에서 돌고 있는가

노트북 코드는 `from cap_lab... import ...`로 이 과정의 도우미를 가져옵니다. `cap_lab`은 `Day04_new/cap_lab/`에 있는 패키지이고, 커널의 작업 폴더가 `Day04_new`일 때만 import됩니다. 또 `.env`와 `outputs/` 같은 상대 경로도 이 폴더를 기준으로 합니다.

아래 셀은 작업 폴더와 `sys.prefix`(지금 쓰는 파이썬 환경의 루트)를 출력합니다. `sys.prefix`는 저장소 루트의 `.venv`여야 합니다. 이 저장소는 루트에 `.venv`가 하나뿐인 uv workspace이고, Day04_new에는 자체 `pyproject.toml`이 없습니다. 다른 환경에서 커널이 떠 있으면 패키지 버전이 달라 이후 셀이 다르게 동작할 수 있습니다.

In [ ]:
# [따라하기] 커널 작업 폴더, 파이썬 환경, cap_lab import 확인
import sys
from pathlib import Path

# 1) 커널이 지금 서 있는 폴더와 파이썬 환경의 루트를 출력한다.
working_folder = Path.cwd()
print("작업 폴더:", working_folder)
print("sys.prefix:", sys.prefix)

# 2) 폴더 이름이 Day04_new가 아니면 이후 상대 경로가 모두 어긋나므로 여기서 멈춘다.
if working_folder.name != "Day04_new":
    raise RuntimeError(
        f"작업 폴더가 Day04_new가 아닙니다: {working_folder.name}. "
        "Day04_new 폴더에서 Jupyter를 다시 여세요."
    )

# 3) cap_lab을 import하고, 어느 파일에서 가져왔는지 출력한다.
import cap_lab
print("cap_lab 위치:", Path(cap_lab.__file__).resolve())
print("폴더 확인: 통과")

## 2. 키는 값이 아니라 상태만 확인한다

이 과정은 키를 네 군데에 씁니다.

| 환경 변수 | 어디에 쓰는가 | 필요한 시점 |
|---|---|---|
| `ANTHROPIC_API_KEY` | Deep Agent의 주 모델 Claude | 01부터 |
| `TYPESAFE_API_KEY` | Jev(TypeSafe Decision API)에 판단을 맡길 때 | 01부터 |
| `LANGFUSE_PUBLIC_KEY`, `LANGFUSE_SECRET_KEY`, `LANGFUSE_HOST` | 측정 기록을 Langfuse에 남길 때 | 07 |

`key_table()`(`cap_lab/notebook.py:59`)은 위 다섯 이름을 **설정됨 / 미설정**으로만 보여 줍니다. 값, 앞자리, 길이는 보여 주지 않습니다(`cap_lab/settings.py:75`의 `key_status`와 같은 규칙). 키는 `Day04_new/.env`에서만 읽습니다(`cap_lab/settings.py:50`의 `load_settings`). 셸에 남아 있는 다른 키로 모르게 과금되는 일을 막으려는 설계입니다.

**미설정이 보이면** 직접 채웁니다. 이 노트북이나 대화창에 키를 붙여 넣지 마세요.

1. `Day04_new/.env.example`을 `Day04_new/.env`로 복사합니다.
2. `.env`를 편집기로 열어 해당 줄의 `=` 뒤에 본인의 키를 적고 저장합니다.
3. 커널을 재시작(Restart)하고 이 셀을 다시 실행합니다. 이미 떠 있는 커널은 `.env`를 다시 읽지 않습니다.

Decision API 키는 `.env.example`에 TYPESAFE, OPENROUTER, OPENAI, AZURE 네 가지가 있고 하나만 있으면 됩니다. 이 노트북은 TYPESAFE 기준으로 표를 보여 줍니다. 다른 공급자를 쓸 때는 `DAY04_DECISION_BACKEND`로 고릅니다(`cap_lab/decisions.py:184`). `.env`는 커밋하지 않는 파일입니다.

In [ ]:
# [따라하기] 키 설정 상태 확인 (값은 출력하지 않는다)
from cap_lab.notebook import key_table

# 1) Day04_new/.env 를 읽어 다섯 이름의 설정됨/미설정만 표로 보여 준다.
key_table()

# 2) ANTHROPIC_API_KEY 와 TYPESAFE_API_KEY 는 01부터, LANGFUSE 세 값은 07에서 필요하다.
#    LANGFUSE 가 미설정이어도 00~06은 진행할 수 있다.
print("ANTHROPIC, TYPESAFE 는 01부터, LANGFUSE 세 값은 07에서 필요합니다.")

## 3. MCP 서버를 띄우고 Tool 7개를 본다

과정 전체에서 쓰는 요구사항 인터뷰 MCP 서버는 `cap_lab/server/server.py`에 있고, `connect_server`(`cap_lab/notebook.py:77`)가 이 서버를 **자식 프로세스로 띄워 stdio로 연결**한 `mcp.Client`를 줍니다. 서버의 세션 DB와 근거(Evidence) 파일은 `outputs/notebook/<이름>-<날짜-시각>/` 아래에 쓰입니다. 이 폴더는 커밋하지 않습니다.

서버가 내놓는 Tool은 7개입니다.

| Tool | 하는 일 |
|---|---|
| `start_interview` | 인터뷰 세션을 시작한다(상태 `draft`) |
| `record_answer` | 질문 항목에 대한 답을 기록한다 |
| `get_session` | 세션의 현재 내용을 읽는다 |
| `freeze_seed` | 요구사항 시드를 확정한다(상태 `frozen`) |
| `list_evidence` | 근거 목록을 본다 |
| `read_evidence` | 근거 하나의 원문을 읽는다 |
| `save_evidence` | 근거를 적재한다 |

세션 상태는 `draft` -> `ready` -> `frozen` 순으로 갑니다. Tool을 호출하지 않고 **목록만** 요청하므로 모델 호출이 없고 비용이 들지 않습니다.

In [ ]:
# [따라하기] 서버를 띄우고 Tool 목록만 읽는다 (모델 호출 없음)
from cap_lab.notebook import connect_server, new_workdir, show_table

# 1) 이번 점검 전용 작업 폴더. 서버의 DB와 근거 파일이 이 아래에 생긴다.
workdir = new_workdir("setup-check")
print("작업 폴더:", workdir)

# 2) stdio 로 서버를 띄워 연결한다. 이 셀은 최상위 await 를 쓴다(Jupyter 는 허용).
async with connect_server(workdir) as client:
    # 3) tools/list 요청. 응답의 tools 에 Tool 정의가 들어 있다.
    tool_list = await client.list_tools()

# 4) 이름과 설명의 첫 줄만 표로 본다.
tool_rows = [
    {"Tool": tool.name, "설명(첫 줄)": (tool.description or "").strip().splitlines()[0] if tool.description else ""}
    for tool in tool_list.tools
]
show_table(tool_rows)

# 5) 7개가 아니면 서버가 제대로 뜬 것이 아니므로 여기서 멈춘다.
tool_names = [tool.name for tool in tool_list.tools]
print("Tool 개수:", len(tool_names))
assert len(tool_names) == 7, f"Tool 이 7개여야 합니다: {tool_names}"

## 4. 클라이언트와 서버가 실제로 쓰는 MCP 프로토콜 버전

MCP는 날짜로 버전을 부릅니다. 어느 버전으로 이야기하는지에 따라 연결 방식 자체가 다릅니다. 2026-07-28 버전에서는 `initialize` 핸드셰이크가 없어지고(stateless), 대신 `server/discover` 요청으로 서버가 지원하는 버전 목록을 받습니다. 그러니 "mcp 패키지가 2.3.0이다"는 답이 아닙니다. 패키지 버전과 프로토콜 버전은 별개이고, **지금 연결에서 협상된 값**을 읽어야 합니다.

설치된 mcp 2.3.0에서 확인한 읽는 길은 다음과 같습니다(추측이 아니라 코드를 읽고 확인).

- `client.protocol_version`: 이 연결에서 협상된 버전 (`.venv/lib/python3.13/site-packages/mcp/client/client.py:502`)
- `client.session.discover_result.supported_versions`: 서버가 `server/discover`에 답한 지원 버전 목록 (`.venv/lib/python3.13/site-packages/mcp/client/session.py:801`). `initialize` 방식으로 연결됐다면 `None`입니다.
- `client.session.initialize_result`: 옛 방식(initialize)으로 연결됐을 때만 값이 있습니다 (`.venv/lib/python3.13/site-packages/mcp/client/session.py:796`)

세 값을 읽을 수 없는 경우에는 짐작한 값을 쓰지 않고 `확인 불가`와 이유를 출력하도록 도우미를 짰습니다.

In [ ]:
# [따라하기] 프로토콜 버전 측정: 협상된 값과 서버가 지원한다고 답한 값을 읽는다
from importlib.metadata import version

from cap_lab.notebook import connect_server, new_workdir


def read_protocol_info(client) -> dict:
    """연결된 mcp.Client 에서 프로토콜 버전을 읽어 사전으로 돌려준다.

    읽을 수 없는 값은 짐작하지 않고 '확인 불가: 이유' 문자열로 둔다.
    """
    info = {}

    # 1) 이 연결에서 협상된 버전
    try:
        info["협상된 버전(client.protocol_version)"] = client.protocol_version
    except (AttributeError, RuntimeError) as error:
        info["협상된 버전(client.protocol_version)"] = f"확인 불가: {type(error).__name__}"

    # 2) 서버가 server/discover 로 알려 준 지원 버전 목록
    discover_result = getattr(client.session, "discover_result", None)
    if discover_result is None:
        info["서버 지원 버전(discover_result)"] = "확인 불가: discover 로 연결되지 않았다(initialize 방식이거나 아직 연결 전)"
    else:
        info["서버 지원 버전(discover_result)"] = list(discover_result.supported_versions)

    # 3) 옛 방식의 initialize 결과는 있으면 보여 주고, 없으면 없다고 말한다.
    info["initialize_result"] = "있음" if client.session.initialize_result is not None else "없음(initialize 를 하지 않았다)"

    # 4) 서버가 스스로 밝힌 이름과 버전. 밝히지 않았으면 None 이다.
    server_info = client.server_info
    info["서버 이름"] = server_info.name if server_info is not None else "확인 불가: 서버가 밝히지 않았다"
    return info


protocol_workdir = new_workdir("setup-protocol")
async with connect_server(protocol_workdir) as client:
    protocol_info = read_protocol_info(client)

print("mcp 패키지 버전:", version("mcp"))
for label, value in protocol_info.items():
    print(f"{label}: {value}")

위 출력에서 `협상된 버전`이 이 수업이 쓰는 프로토콜 버전입니다. 과정의 설계 메모(`docs/DESIGN.md`)는 이 값을 `unknown`으로 남겨 두었고, 이 셀이 그것을 실제 연결에서 읽어 채웁니다. 03과 04에서 stateless, `_meta`, MRTR을 이야기할 때 이 버전이 전제가 됩니다.

## 5. Claude 한 번 불러 보기 (유료)

이제 첫 유료 호출입니다. `make_model(load_settings())`(`cap_lab/settings.py:90`)는 `ChatAnthropic`을 만듭니다. 모델과 effort는 `.env`의 `DAY04_MODEL`, `DAY04_EFFORT`에서 오고, 기본은 `claude-sonnet-5-5`와 `medium`입니다. `temperature`와 thinking 옵션은 보내지 않고, `max_retries=0`이라 일시적 429도 숨기지 않고 실패로 드러납니다.

**먼저 예측하세요.** 한 문장짜리 질문에 대한 `usage_metadata`의 입력 토큰은 대략 몇 개일까요? 출력 토큰은 답의 글자 수보다 많을까요, 적을까요? (effort가 `medium`이라는 점을 생각해 보세요.) 숫자를 노트에 적은 뒤 실행합니다.

In [ ]:
# [따라하기] Claude 한 번 호출 (유료)
from cap_lab.notebook import announce
from cap_lab.settings import load_settings, make_model

# 1) 실행 직전에 호출 규모를 알린다. Claude 1회.
announce("Claude 연결 확인", model_calls=1)

# 2) .env 에서 설정을 읽고 ChatAnthropic 을 만든다. ANTHROPIC_API_KEY 가 없으면 여기서 멈춘다.
settings = load_settings()
model = make_model(settings)
print("모델:", settings.model, "/ effort:", settings.effort)

# 3) 한 문장짜리 프롬프트로 한 번 호출한다.
reply = model.invoke("MCP가 무엇인지 한 문장으로 설명하세요.")

# 4) 답과 토큰 사용량을 출력한다. usage_metadata 는 없을 수도 있다.
print("답:", reply.text)
usage = reply.usage_metadata or {}
print("입력 토큰:", usage.get("input_tokens"))
print("출력 토큰:", usage.get("output_tokens"))
print("합계 토큰:", usage.get("total_tokens"))

**관찰할 것.** (1) 답이 한 문장으로 나왔는지, (2) 출력 토큰이 답의 글자 수보다 큰지 작은지. 예측과 다르다면 effort가 `medium`일 때 모델이 답 앞에 추론 토큰을 쓸 수 있다는 점이 이유일 수 있습니다. 이 값이 확인되지 않으면 "확인 필요"로 적어 두세요. (3) 실패했다면 오류 클래스 이름이 무엇인지. 키 미설정이면 `RuntimeError`, 잘못된 키면 인증 오류가 나옵니다. 이 호출은 1회 관찰이므로 토큰 수를 일반화하지 마세요.

## 6. Jev 한 번 불러 보기 (유료)

Jev는 **글을 써 주는 모델이 아니라 질문에 확률로 답하는 판단 모델**입니다. Noul 질문은 참/거짓 질문이고, 질문 문장(`instructions`)과 참이면/거짓이면의 기준 문장(`true`, `false`)을 함께 보냅니다(`cap_lab/decisions.py:26`의 `Noul`). 답은 `Judgment`(`cap_lab/decisions.py:70`)의 `verdicts[질문이름].probability`에 **0에서 1 사이 숫자**로 옵니다. 이 숫자는 "참일 확률"이지 정답이 아닙니다. 문턱값(예: 0.5)을 어디에 두는지는 호출하는 쪽의 몫이고, 보정(calibration)을 가정하지 않습니다.

여기서는 과정에서 실제로 쓰는 질문 `GOAL_PRESERVED`(`cap_lab/middleware.py:25`)를 그대로 써서 연결만 확인합니다. 사용자가 말한 목표와 `start_interview`의 `goal` 인자를 받아, goal이 목표를 그대로 담고 있는지 묻는 질문입니다. 상태 문장의 모양은 `cap_lab/middleware.py:80`과 같습니다. 질문과 상태 문장은 선택한 Decision API 공급자에게 그대로 전송됩니다.

**먼저 예측하세요.** 사용자는 "월간 매출 보고서를 자동으로 만들고 싶다"고 말했고, goal 인자는 "보고서 자동화 명세 만들기"입니다. 확률이 0.5보다 클까요, 작을까요? 1에 가까울까요, 0에 가까울까요?

In [ ]:
# [바꿔보기] Jev 한 번 호출 (유료). 상태 문장의 goal 을 바꿔 보며 확률이 어떻게 달라지는지 본다.
from cap_lab.decisions import make_decider
from cap_lab.middleware import GOAL_PRESERVED
from cap_lab.notebook import announce
from cap_lab.settings import load_settings

# 1) 실행 직전에 호출 규모를 알린다. Claude 0회, Jev 1회.
announce("Jev 연결 확인", model_calls=0, jev_calls=1)

# 2) 키가 있는 Decision API 공급자를 골라 판단기를 만든다. 키가 없으면 여기서 멈춘다.
decider = make_decider(load_settings())
print("공급자:", decider.name)

# 3) 상태 문장: cap_lab/middleware.py:80 과 같은 모양이다. 아래 goal 문장을 바꿔 보면 확률이 달라진다.
user_request = "월간 매출 보고서를 자동으로 만들고 싶다"
goal_argument = "보고서 자동화 명세 만들기"
state = f"사용자 요청:\n{user_request}\n\nstart_interview의 goal 인자:\n{goal_argument}"

# 4) 질문 하나를 보내 판정을 받는다.
judgment = decider.judge(state, [GOAL_PRESERVED])
verdict = judgment.verdicts[GOAL_PRESERVED.name]

# 5) 확률(참일 가능성)을 읽는다. 모델이 판정을 거절했으면 refused 이고 확률은 None 이다.
print("거절 여부:", verdict.refused)
print("goal 이 그대로일 확률:", verdict.probability)
print("입력/출력 토큰:", judgment.input_tokens, "/", judgment.output_tokens)

**관찰할 것.** (1) 출력은 문장이 아니라 0과 1 사이의 숫자 하나입니다. 같은 입력으로 다시 실행하면 값이 조금 달라질 수 있으므로 한 번의 값으로 결론을 내지 마세요(이 실행은 1회 관찰입니다). (2) 예측한 방향(0.5보다 큰지 작은지)이 맞았는지. (3) `refused`가 `True`였다면 확률이 `None`이라는 점. 이 과정의 `GoalGuard`는 판정하지 못한 경우를 통과로 바꾸지 않고 `decision_unavailable`로 막습니다. 그 이유는 04에서 다룹니다.

## 7. Langfuse 인증 확인 (모델 호출 없음)

07의 측정 기록은 Langfuse에 남습니다. 지금은 키가 유효한지만 봅니다. `auth_check()`는 Langfuse 서버에 인증을 시험하는 요청이고 모델을 부르지 않습니다. 키 값은 출력하지 않고 결과만 `ok` 또는 `fail: <예외 클래스 이름>`으로 보여 줍니다.

**실패해도 00~06은 진행할 수 있습니다.** 07에서는 Langfuse가 반드시 필요하니, `fail: UnauthorizedError`처럼 인증이 거절되면 강사에게 키 교체를 요청하세요. 키가 미설정이면 `fail: 키 미설정`이 나옵니다.

In [ ]:
# [따라하기] Langfuse 인증 확인 (모델 호출 없음, 키는 출력하지 않는다)
from langfuse import Langfuse

from cap_lab.settings import load_settings

settings = load_settings()

# 1) 공개키와 비밀키가 둘 다 있어야 시험할 수 있다. 값은 출력하지 않는다.
if not (settings.langfuse_public_key.strip() and settings.langfuse_secret_key.strip()):
    langfuse_status = "fail: 키 미설정"
else:
    # 2) 클라이언트를 만들고 인증을 시험한다. host 가 비어 있으면 Langfuse 기본 주소를 쓴다.
    langfuse_client = Langfuse(
        public_key=settings.langfuse_public_key,
        secret_key=settings.langfuse_secret_key,
        host=settings.langfuse_host or None,
    )
    try:
        langfuse_status = "ok" if langfuse_client.auth_check() else "fail: auth_check"
    except Exception as error:  # 인증 거절, 네트워크 오류 등. 클래스 이름만 남긴다.
        langfuse_status = f"fail: {type(error).__name__}"
    finally:
        langfuse_client.shutdown()

print("Langfuse 인증:", langfuse_status)
print("07에서 Langfuse 가 필요합니다. fail 이면 키를 교체한 뒤 이 셀을 다시 실행하세요.")

## 직접 해보기

앞의 셀을 실행한 뒤, 아래 세 가지를 직접 해 보세요. 1번과 2번은 비용이 없고, 3번은 Jev를 한 번 부르므로 `RUN_EXERCISE = False`로 잠겨 있습니다. 켜는 것은 본인의 결정입니다.

### 해보기 1. 키가 하나도 없을 때의 모습을 본다

`load_settings`는 가리킨 `.env` 파일이 없으면 아무것도 읽지 않습니다(`cap_lab/settings.py:50`의 docstring). 존재하지 않는 경로를 넘기면 키가 하나도 없는 상태를 `.env`를 건드리지 않고 재현할 수 있습니다. 표가 어떻게 바뀌는지 보고, 왜 이 상태에서 01을 시작하면 안 되는지 말해 보세요.

### 해보기 2. Tool 하나의 입력 스키마를 읽는다

3절에서 받은 `tool_list`에서 `start_interview`의 입력 스키마를 출력하고, 어떤 인자가 필수이고 모델이 채워야 하는지 찾아보세요. 02에서 이 스키마가 모델의 Tool 선택과 인자 채우기에 어떻게 영향을 주는지 다룹니다.

### 해보기 3. goal 을 바꿔 확률을 비교한다 (유료, 잠금)

6절의 `goal_argument`를 사용자의 말과 똑같은 문장으로 바꿔 한 번 더 판정해 보세요. 확률이 어느 방향으로 움직였는지 적고, 두 번 모두 1회 관찰이라는 점을 기억하세요. 같은 질문으로 3회 이상 반복해야 흔들림이 보입니다.

In [ ]:
# [바꿔보기] 해보기 1: 키가 하나도 없는 상태의 표 (비용 없음)
from cap_lab.notebook import key_table
from cap_lab.settings import load_settings

# 존재하지 않는 .env 경로를 가리키면 아무것도 읽지 않는다. 실제 .env 는 건드리지 않는다.
empty_settings = load_settings("no-such-file.env")
key_table(empty_settings)

In [ ]:
# [바꿔보기] 해보기 2: start_interview 의 입력 스키마 읽기 (비용 없음)
import json

# 3절에서 받은 tool_list 에서 Tool 하나를 이름으로 고른다.
start_interview_tool = next(tool for tool in tool_list.tools if tool.name == "start_interview")
print("Tool:", start_interview_tool.name)

# inputSchema 는 JSON Schema 다. 필수 인자(required)와 각 인자의 타입을 본다.
input_schema = start_interview_tool.input_schema
print("필수 인자:", input_schema.get("required"))
print(json.dumps(input_schema.get("properties"), ensure_ascii=False, indent=2))

In [ ]:
# [바꿔보기] 해보기 3: goal 을 사용자 말 그대로 바꿔 다시 판정 (유료, 기본은 잠겨 있다)
RUN_EXERCISE = False

if RUN_EXERCISE:
    from cap_lab.decisions import make_decider
    from cap_lab.middleware import GOAL_PRESERVED
    from cap_lab.notebook import announce
    from cap_lab.settings import load_settings

    announce("해보기 3: goal 바꿔 판정", model_calls=0, jev_calls=1)
    exercise_decider = make_decider(load_settings())
    same_goal = "월간 매출 보고서를 자동으로 만들고 싶다"  # 사용자의 말과 같은 문장
    exercise_state = f"사용자 요청:\n{user_request}\n\nstart_interview의 goal 인자:\n{same_goal}"
    exercise_verdict = exercise_decider.judge(exercise_state, [GOAL_PRESERVED]).verdicts[GOAL_PRESERVED.name]
    print("goal 이 그대로일 확률(1회 관찰):", exercise_verdict.probability)
else:
    print("RUN_EXERCISE = False 입니다. Jev 를 부르지 않았습니다.")

## 확인 질문

1. `sys.prefix`가 저장소 루트의 `.venv`가 아니라면 어떤 문제가 생길 수 있고, `key_table()`이 값을 보여 주지 않는 이유는 무엇입니까?
2. `client.protocol_version`과 `version("mcp")`는 각각 무엇을 말합니까? 둘이 다른 이유를 설명해 보세요.
3. Jev의 답 `0.8`은 "goal이 맞다"는 뜻입니까? 문턱값을 0.5로 둔다면 어떤 결정이 내려지고, 그 값이 믿을 만한지는 어떻게 알 수 있습니까?
4. `Langfuse` 인증이 실패해도 02 노트북은 진행할 수 있지만 07은 진행할 수 없습니다. 이유를 이 노트북의 내용으로 설명해 보세요.

## 정리

- 커널의 작업 폴더는 `Day04_new`여야 하고 `cap_lab`이 import되어야 합니다.
- 키는 `Day04_new/.env`에서만 읽고, 설정됨/미설정으로만 확인합니다. 미설정은 수강생이 `.env.example`을 보고 직접 채웁니다.
- MCP 서버는 stdio로 뜨고 Tool 7개가 보입니다. 프로토콜 버전은 패키지 버전이 아니라 **연결에서 협상된 값**을 읽습니다.
- Claude는 글을 쓰는 모델이고, Jev는 질문에 확률로 답하는 판단 모델입니다. 확률은 정답이 아니며 문턱값은 사용하는 쪽이 정합니다.
- Langfuse 인증은 07에서 필요합니다. 인증이 실패하면 키를 교체한 뒤 다시 확인하세요.
- 유료 호출은 이 노트북에서 Claude 1회, Jev 1회였고, 모든 유료 셀은 실행 직전에 `announce`로 규모를 알렸습니다.